# 12 - Silver Order Items

Explodes the latest order state and synchronizes `(order_id, sku)` items, preserving original and USD prices.

In [ ]:
%run ./05_bronze_common

In [ ]:
import uuid
from datetime import datetime, timezone

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window


def run_version(run_id: str) -> int:
    return int(run_id.replace("-", "")[:15], 16)


def merge_quarantine(table_name: str, errors) -> int:
    count = errors.count()
    if count:
        (
            DeltaTable.forName(spark, table_name)
            .alias("target")
            .merge(errors.alias("source"), "target.error_id = source.error_id")
            .whenNotMatchedInsertAll()
            .execute()
        )
    return count


def silver_dq(
    catalog: str,
    source_name: str,
    pipeline_run_id: str,
    valid_count: int,
    errors,
    duplicate_count: int = 0,
) -> None:
    error_counts = errors.groupBy("error_code").count().collect()
    if duplicate_count:
        error_counts.append(Row(error_code="duplicate_key", count=duplicate_count))
    now = datetime.now(timezone.utc)
    rows = [
        Row(
            quality_date=now.date(),
            source_name=source_name,
            pipeline_stage="silver",
            quality_status="valid",
            error_code=None,
            record_count=int(valid_count),
            pipeline_run_id=pipeline_run_id,
            recorded_ts=now,
        )
    ]
    rows.extend(
        Row(
            quality_date=now.date(),
            source_name=source_name,
            pipeline_stage="silver",
            quality_status="error",
            error_code=row.error_code,
            record_count=int(row["count"]),
            pipeline_run_id=pipeline_run_id,
            recorded_ts=now,
        )
        for row in error_counts
    )
    append_delta(
        spark.createDataFrame(rows, DQ_SCHEMA),
        f"{catalog}.control.data_quality_results",
        f"{source_name}_silver_dq",
        run_version(pipeline_run_id),
    )


def silver_error(
    error_code,
    error_reason,
    record_key,
    raw_payload,
    rescued_data,
    source_file,
    batch_id,
    load_date,
    ingest_ts,
    pipeline_run_id,
):
    return [
        F.sha2(
            F.concat_ws(
                "||",
                source_file,
                F.coalesce(record_key, F.lit("")),
                raw_payload,
                error_code,
            ),
            256,
        ).alias("error_id"),
        F.lit("silver").alias("error_stage"),
        error_code.alias("error_code"),
        error_reason.alias("error_reason"),
        record_key.alias("record_key"),
        raw_payload.alias("raw_payload"),
        rescued_data.alias("rescued_data"),
        source_file.alias("source_file"),
        batch_id.alias("batch_id"),
        load_date.alias("load_date"),
        ingest_ts.alias("ingest_ts"),
        F.current_date().alias("quarantine_date"),
        pipeline_run_id.alias("pipeline_run_id"),
    ]

In [ ]:
dbutils.widgets.text("catalog", "abc_retail", "Target catalog")
dbutils.widgets.text("job_run_id", "manual", "Notebook or job run ID")
catalog = validate_identifier(dbutils.widgets.get("catalog").strip(), "catalog")
job_run_id = dbutils.widgets.get("job_run_id").strip()
spark.conf.set("spark.sql.session.timeZone", "UTC")

pipeline_name = "silver_order_items"
pipeline_run_id = str(uuid.uuid4())
start_ts = datetime.now(timezone.utc)
orders_table = f"{catalog}.silver.silver_orders"
target_table = f"{catalog}.silver.silver_order_items"

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS `{catalog}`.`silver`.`silver_order_items` (
        order_id STRING NOT NULL,
        sku STRING NOT NULL,
        qty BIGINT NOT NULL,
        unit_price_original DECIMAL(18,2) NOT NULL,
        currency STRING NOT NULL,
        fx_rate_to_usd DECIMAL(20,8) NOT NULL,
        unit_price_usd DECIMAL(18,2) NOT NULL,
        line_total_original DECIMAL(24,2) NOT NULL,
        line_total_usd DECIMAL(24,2) NOT NULL,
        order_updated_at TIMESTAMP NOT NULL,
        ingest_ts TIMESTAMP NOT NULL,
        source_file STRING NOT NULL,
        batch_id STRING NOT NULL,
        pipeline_run_id STRING NOT NULL
    )
    USING DELTA
    TBLPROPERTIES ('comment' = 'Latest-state order items; Bronze preserves prior versions')
    """
)

upsert_audit(catalog, pipeline_run_id, pipeline_name, "orders_latest_state", start_ts, "RUNNING", [orders_table], job_run_id)
try:
    orders = spark.table(orders_table)
    current = orders.select(
        "order_id", "currency", "fx_rate_to_usd", "updated_at", "ingest_ts",
        "source_file", "batch_id", F.explode("items").alias("item"),
    ).select(
        "order_id", F.col("item.sku").alias("sku"), F.col("item.qty").alias("qty"),
        F.col("item.unit_price").cast("decimal(18,2)").alias("unit_price_original"),
        "currency", "fx_rate_to_usd",
        (F.col("item.unit_price") * F.col("fx_rate_to_usd")).cast("decimal(18,2)").alias("unit_price_usd"),
        (F.col("item.qty") * F.col("item.unit_price")).cast("decimal(24,2)").alias("line_total_original"),
        (F.col("item.qty") * F.col("item.unit_price") * F.col("fx_rate_to_usd")).cast("decimal(24,2)").alias("line_total_usd"),
        F.col("updated_at").alias("order_updated_at"), "ingest_ts", "source_file", "batch_id",
        F.lit(pipeline_run_id).alias("pipeline_run_id"),
    )
    target_df = spark.table(target_table)
    stale = target_df.select("order_id", "sku").join(current.select("order_id", "sku"), ["order_id", "sku"], "left_anti").withColumn("_delete", F.lit(True))
    source = current.withColumn("_delete", F.lit(False)).unionByName(stale, allowMissingColumns=True)
    (
        DeltaTable.forName(spark, target_table).alias("target")
        .merge(source.alias("source"), "target.order_id = source.order_id AND target.sku = source.sku")
        .whenMatchedDelete(condition="source._delete")
        .whenMatchedUpdateAll(condition="NOT source._delete AND (source.order_updated_at > target.order_updated_at OR source.qty <> target.qty OR source.unit_price_original <> target.unit_price_original OR source.fx_rate_to_usd <> target.fx_rate_to_usd)")
        .whenNotMatchedInsertAll(condition="NOT source._delete")
        .execute()
    )
    count = current.count()
    upsert_audit(catalog, pipeline_run_id, pipeline_name, "orders_latest_state", start_ts, "SUCCESS", [orders_table], job_run_id, end_ts=datetime.now(timezone.utc), rows_read=orders.count(), rows_written=count, rows_quarantined=0)
except Exception as exc:
    upsert_audit(catalog, pipeline_run_id, pipeline_name, "orders_latest_state", start_ts, "FAILED", [orders_table], job_run_id, end_ts=datetime.now(timezone.utc), error_message=str(exc)[:4000])
    raise